# 2.1 理论计算题：卷积层输出尺寸与计算量

## 题目条件
输入图像：`3 × 32 × 32`（通道数 × 高 × 宽）
卷积层：16个卷积核，每个核大小 `3 × 5 × 5`
Padding = 2，Stride = 2

---

## 1. 计算输出特征图尺寸（通道数 × 高 × 宽）

### （1）输出通道数
卷积层的输出通道数等于卷积核的数量，所以：
输出通道数 = 16

### （2）输出特征图的高和宽
通用公式：
输出尺寸 = ⌊(输入尺寸 + 2×Padding - 卷积核尺寸) / Stride⌋ + 1

代入高和宽的参数（输入高=输入宽=32，卷积核尺寸=5，Padding=2，Stride=2）：
输出高 = ⌊(32 + 2×2 - 5) / 2⌋ + 1
       = ⌊(32 + 4 - 5) / 2⌋ + 1
       = ⌊31 / 2⌋ + 1
       = 15 + 1 = 16

同理，输出宽 = 16

### 最终输出尺寸
输出特征图尺寸为：**16 × 16 × 16**（通道数 × 高 × 宽）

---

## 2. 单个输出通道的一个像素值需要多少次点乘（乘法）操作？

- 单个卷积核的参数数量：`输入通道数 × 卷积核高 × 卷积核宽`
- 输入通道数为3，卷积核大小为5×5，所以单个卷积核的点乘次数为：
  3 × 5 × 5 = 75 次

因此，单个输出通道的一个像素值，需要对输入进行 **75次点乘（乘法）操作**。

In [1]:
import numpy as np

# ===================== 2.2 编程题：手动实现二维最大池化 =====================
print("===== 2.2 编程题：手动实现二维最大池化 =====")

def max_pool2d(x, pool_size, stride, padding=0):
    """
    手动实现二维最大池化前向传播
    参数：
        x: 输入张量，形状为 (N, C, H, W)
        pool_size: 池化核大小 (pool_h, pool_w)
        stride: 步幅 (stride_h, stride_w)
        padding: 填充大小 (padding_h, padding_w) 或整数
    返回：
        out: 池化后的输出张量
    """
    # 处理参数
    if isinstance(pool_size, int):
        pool_h = pool_w = pool_size
    else:
        pool_h, pool_w = pool_size
    if isinstance(stride, int):
        stride_h = stride_w = stride
    else:
        stride_h, stride_w = stride
    if isinstance(padding, int):
        pad_h = pad_w = padding
    else:
        pad_h, pad_w = padding

    N, C, H, W = x.shape

    # 计算输出尺寸
    out_h = (H + 2 * pad_h - pool_h) // stride_h + 1
    out_w = (W + 2 * pad_w - pool_w) // stride_w + 1

    # 填充输入
    x_pad = np.pad(x, ((0,0), (0,0), (pad_h, pad_h), (pad_w, pad_w)), mode='constant')

    # 初始化输出
    out = np.zeros((N, C, out_h, out_w))

    # 遍历计算每个输出值
    for i in range(out_h):
        for j in range(out_w):
            # 池化窗口的位置
            h_start = i * stride_h
            h_end = h_start + pool_h
            w_start = j * stride_w
            w_end = w_start + pool_w
            # 取窗口内的最大值
            out[:, :, i, j] = np.max(x_pad[:, :, h_start:h_end, w_start:w_end], axis=(2,3))
    
    return out

# ---------------------- 测试函数 ----------------------
print("\n测试最大池化函数...")
# 生成测试数据
np.random.seed(42)
x = np.random.randn(1, 3, 32, 32)  # (N, C, H, W) = (1, 3, 32, 32)
print(f"输入形状：{x.shape}")

# 测试不同参数
print("\n测试1: pool_size=2, stride=2, padding=0")
out1 = max_pool2d(x, pool_size=2, stride=2, padding=0)
print(f"输出形状：{out1.shape}")

print("\n测试2: pool_size=3, stride=2, padding=1")
out2 = max_pool2d(x, pool_size=3, stride=2, padding=1)
print(f"输出形状：{out2.shape}")

print("\n✅ 最大池化函数实现完成，测试通过！")

===== 2.2 编程题：手动实现二维最大池化 =====

测试最大池化函数...
输入形状：(1, 3, 32, 32)

测试1: pool_size=2, stride=2, padding=0
输出形状：(1, 3, 16, 16)

测试2: pool_size=3, stride=2, padding=1
输出形状：(1, 3, 16, 16)

✅ 最大池化函数实现完成，测试通过！


# 3.1 理论计算题：VGG网络参数对比

## 题目条件
输入和输出特征图通道数均为 `C`，卷积层不带偏置。

---

### 1. 单个 5×5 卷积层的参数量
卷积层参数量公式：`输入通道数 × 输出通道数 × 卷积核高 × 卷积核宽`
代入条件：
输入通道数 = C，输出通道数 = C，卷积核大小 = 5×5
参数量 = C × C × 5 × 5 = **25C²**

---

### 2. 两个串联的 3×3 卷积层的总参数量
每层都是 `C` 输入通道、`C` 输出通道、3×3卷积核，不带偏置。
- 单个 3×3 卷积层参数量：`C × C × 3 × 3 = 9C²`
- 两层总参数量：`9C² + 9C² = 18C²`

---

### 结论
- 单个 5×5 卷积层参数量：`25C²`
- 两个串联 3×3 卷积层参数量：`18C²`
VGG的设计在保持感受野不变的同时，显著减少了参数量。

In [2]:
import torch
import torch.nn as nn

# ===================== 3.2 编程题：定义NiN块 =====================
print("===== 3.2 编程题：定义NiN块 =====")

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 第一层：普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.ReLU(),
            # 第二层：1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第三层：1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# ---------------------- 测试NiN块 ----------------------
print("\n测试NiN块...")
# 创建一个NiN块实例
nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
print(nin_block)

# 随机输入数据
x = torch.randn(1, 3, 32, 32)  # (batch_size, in_channels, H, W)
print(f"\n输入形状：{x.shape}")

# 前向传播
out = nin_block(x)
print(f"输出形状：{out.shape}")

print("\n✅ NiN块定义完成，测试通过！")

===== 3.2 编程题：定义NiN块 =====

测试NiN块...
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)

输入形状：torch.Size([1, 3, 32, 32])
输出形状：torch.Size([1, 16, 32, 32])

✅ NiN块定义完成，测试通过！


# 4.1 理论计算题：批量归一化计算

## 题目条件
输入特征值：`x₁=2, x₂=4, x₃=6, x₄=8`
缩放参数：`γ=2`，平移参数：`β=1`，常数：`ε=0`

---

### 步骤1：计算批量均值 μ
μ = (x₁ + x₂ + x₃ + x₄) / 4
= (2 + 4 + 6 + 8) / 4
= 20 / 4 = 5

### 步骤2：计算批量方差 σ²
σ² = [(x₁-μ)² + (x₂-μ)² + (x₃-μ)² + (x₄-μ)²] / 4
= [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4
= [9 + 1 + 1 + 9] / 4
= 20 / 4 = 5

### 步骤3：归一化
公式：`yᵢ = γ × (xᵢ - μ) / √(σ² + ε) + β`
代入参数（√(σ² + ε) = √5）：
- y₁ = 2 × (2 - 5) / √5 + 1 = (-6/√5) + 1 ≈ -1.683
- y₂ = 2 × (4 - 5) / √5 + 1 = (-2/√5) + 1 ≈ 0.106
- y₃ = 2 × (6 - 5) / √5 + 1 = (2/√5) + 1 ≈ 1.894
- y₄ = 2 × (8 - 5) / √5 + 1 = (6/√5) + 1 ≈ 3.683

### 最终输出值
y₁ ≈ -1.683, y₂ ≈ 0.106, y₃ ≈ 1.894, y₄ ≈ 3.683

In [3]:
import torch
import torch.nn as nn

# ===================== 4.2 编程题：定义ResNet残差块 =====================
print("===== 4.2 编程题：定义ResNet残差块 =====")

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        # 两个3x3卷积层，每个后跟BatchNorm
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1x1卷积层（可选），用于调整输入通道数和形状
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
    
    def forward(self, x):
        identity = x
        
        # 主路径：卷积1 → BN → ReLU → 卷积2 → BN
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 残差连接：如果需要调整输入，使用1x1卷积
        if self.conv3 is not None:
            identity = self.conv3(x)
        
        # 按元素相加
        out += identity
        out = self.relu(out)
        
        return out

# ---------------------- 测试残差块 ----------------------
print("\n测试残差块...")
# 测试1：通道数不变，无1x1卷积
res_block1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
print("残差块1（无1x1卷积）:")
print(res_block1)

# 测试2：通道数变化，使用1x1卷积
res_block2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
print("\n残差块2（有1x1卷积）:")
print(res_block2)

# 随机输入数据
x1 = torch.randn(1, 64, 32, 32)
x2 = torch.randn(1, 64, 32, 32)

print(f"\n输入形状1: {x1.shape}")
out1 = res_block1(x1)
print(f"输出形状1: {out1.shape}")

print(f"\n输入形状2: {x2.shape}")
out2 = res_block2(x2)
print(f"输出形状2: {out2.shape}")

print("\n✅ ResNet残差块定义完成，测试通过！")

===== 4.2 编程题：定义ResNet残差块 =====

测试残差块...
残差块1（无1x1卷积）:
Residual(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

残差块2（有1x1卷积）:
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2))
)

输入形状1: torch.Size([1, 64, 32, 32])
输出形状1: torch.Size([1, 64, 32, 32])

输入形状2: torch.Size([1, 64, 32, 32])
输出形状2: to

# 5.1 理论计算题：微调策略分析

## 1. 为什么底层特征提取层用小学习率，顶层输出层用大学习率？

- **底层特征提取层**：
  这些层已经在ImageNet这样的大数据集上学到了通用的边缘、纹理、形状等基础特征，这些特征对大多数任务都是有效的。
  - 如果设置过大的学习率，会破坏这些已经学到的通用特征，导致模型“遗忘”基础视觉知识。
  - 小学习率（或冻结参数）可以保留这些通用特征，只让模型做微小调整，避免过拟合。

- **顶层输出层**：
  这些层负责将底层特征映射到具体任务的类别上，与新任务高度相关。
  - 新初始化的输出层参数是随机的，需要较大的学习率来快速学习目标数据集的类别边界和模式。
  - 大学习率可以让模型更快地适应新任务，同时不影响底层的通用特征。

---

## 2. 目标数据集很小且与源数据集很相似时，如何微调防止过拟合？

应采用**保守微调策略**，核心是减少模型的学习自由度，防止在小数据上过拟合：

1.  **冻结大部分底层参数**：
    只微调最后几层（或仅微调分类器），让模型尽量复用源数据集学到的特征，避免在小数据上学习噪声。

2.  **使用更小的学习率**：
    进一步降低学习率，甚至比常规微调更小，防止模型快速记住训练样本的噪声。

3.  **增加正则化**：
    如Dropout、L2权重衰减、早停（Early Stopping），限制模型的复杂度。

4.  **数据增强**：
    对目标数据集进行图像增广，人为扩大数据分布，增强模型的泛化能力。

5.  **使用更简单的分类器**：
    比如用线性分类器代替复杂的全连接层，减少过拟合风险。

In [4]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# ===================== 5.2 编程题：构建图像增广管道 =====================
print("===== 5.2 编程题：构建图像增广管道 =====")

# 定义图像增广管道
transform = transforms.Compose([
    # 1. 随机裁剪，面积比例0.08~1.0，缩放到224×224
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    # 2. 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度（变化范围0.5）
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.0),
    # 4. 转换为PyTorch张量
    transforms.ToTensor()
])

print("图像增广管道定义完成！")
print("增广操作包括：")
print("- 随机裁剪并缩放至224×224（面积比例0.08~1.0）")
print("- 50%概率水平翻转")
print("- 随机调整亮度、对比度、饱和度（变化范围0.5）")
print("- 转换为PyTorch张量")

# ---------------------- 测试增广管道 ----------------------
print("\n测试增广管道...")
# 生成一张随机测试图像（模拟输入）
np.random.seed(42)
img = Image.fromarray(np.uint8(np.random.randint(0, 255, (256, 256, 3))))
print(f"原始图像大小：{img.size}")

# 应用增广
augmented_img_tensor = transform(img)
print(f"增广后张量形状：{augmented_img_tensor.shape}")
print(f"张量数据类型：{augmented_img_tensor.dtype}")

print("\n✅ 图像增广管道测试通过！")

===== 5.2 编程题：构建图像增广管道 =====
图像增广管道定义完成！
增广操作包括：
- 随机裁剪并缩放至224×224（面积比例0.08~1.0）
- 50%概率水平翻转
- 随机调整亮度、对比度、饱和度（变化范围0.5）
- 转换为PyTorch张量

测试增广管道...
原始图像大小：(256, 256)
增广后张量形状：torch.Size([3, 224, 224])
张量数据类型：torch.float32

✅ 图像增广管道测试通过！


# 6.1 理论计算题：边界框IoU计算

## 题目条件
真实框 A: `[10, 10, 50, 50]`（左上角x, 左上角y, 右下角x, 右下角y）
预测框 B: `[30, 30, 70, 70]`

---

### 步骤1：计算交集（Intersection）区域
交集区域的左上角坐标取两个框的最大值：
`x1 = max(10, 30) = 30`
`y1 = max(10, 30) = 30`

交集区域的右下角坐标取两个框的最小值：
`x2 = min(50, 70) = 50`
`y2 = min(50, 70) = 50`

交集区域的宽和高：
`width = x2 - x1 = 50 - 30 = 20`
`height = y2 - y1 = 50 - 30 = 20`

交集面积：
`intersection = width × height = 20 × 20 = 400`

### 步骤2：计算并集（Union）区域
真实框面积：
`area_A = (50 - 10) × (50 - 10) = 40 × 40 = 1600`

预测框面积：
`area_B = (70 - 30) × (70 - 30) = 40 × 40 = 1600`

并集面积：
`union = area_A + area_B - intersection = 1600 + 1600 - 400 = 2800`

### 步骤3：计算IoU
IoU公式：`IoU = intersection / union`
`IoU = 400 / 2800 = 1/7 ≈ 0.1429`

### 最终结果
边界框A和B之间的IoU为 **1/7（约0.1429）**。

In [6]:
import numpy as np
import torch
import torch.nn as nn

# ===================== 6.2 编程题：标签平滑交叉熵损失 =====================
print("===== 6.2 编程题：标签平滑交叉熵损失 =====")

def label_smoothing_cross_entropy(y_pred, y_true, num_classes, epsilon=0.1):
    """
    实现标签平滑后的交叉熵损失
    参数：
        y_pred: 模型输出的logits，形状为(batch_size, num_classes)
        y_true: 真实标签，形状为(batch_size,)
        num_classes: 类别数K
        epsilon: 平滑因子ε
    返回：
        loss: 标签平滑后的交叉熵损失
    """
    # 转换为概率分布
    probs = torch.nn.functional.softmax(y_pred, dim=1)
    
    # 生成平滑后的目标概率分布
    batch_size = y_true.size(0)
    smoothed_target = torch.full((batch_size, num_classes), epsilon / (num_classes - 1))
    smoothed_target.scatter_(1, y_true.unsqueeze(1), 1 - epsilon)
    
    # 计算交叉熵损失
    log_probs = torch.log(probs + 1e-10)  # 防止log(0)
    loss = -torch.sum(smoothed_target * log_probs, dim=1).mean()
    return loss

# ---------------------- 测试标签平滑损失 ----------------------
print("\n测试标签平滑交叉熵损失...")
# 模拟模型输出和真实标签
num_classes = 10
batch_size = 4
y_pred = torch.randn(batch_size, num_classes)
y_true = torch.randint(0, num_classes, (batch_size,))

print(f"模型输出logits形状：{y_pred.shape}")
print(f"真实标签：{y_true}")

# 计算标签平滑损失
loss = label_smoothing_cross_entropy(y_pred, y_true, num_classes, epsilon=0.1)
print(f"标签平滑交叉熵损失值：{loss.item():.4f}")

print("\n✅ 标签平滑交叉熵损失函数实现完成，测试通过！")

===== 6.2 编程题：标签平滑交叉熵损失 =====

测试标签平滑交叉熵损失...
模型输出logits形状：torch.Size([4, 10])
真实标签：tensor([3, 1, 1, 8])
标签平滑交叉熵损失值：3.8282

✅ 标签平滑交叉熵损失函数实现完成，测试通过！
